In [52]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [54]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [56]:
# Load Report Builder and PromptEvaluator from 001_prompting.ipynb
with open("001_prompting.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        if "class PromptEvaluator" in source or "generate_prompt_evaluation_report" in source:
            exec(source, globals())

evaluator = PromptEvaluator(max_concurrent_tasks=2)
print("PromptEvaluator loaded successfully!")

PromptEvaluator loaded successfully!


In [58]:
dataset = evaluator.generate_dataset(
    task_description="""
    Extract topics out of a passage of text from a scholarly article into a JSON array of strings
    """,
    prompt_inputs_spec={
        "content": "One paragraph of text from a scholarly journal written in English"
    },
    output_file="dataset.json",
    num_cases=4,
)

Generated 1/4 test cases
Generated 2/4 test cases
Generated 3/4 test cases
Generated 4/4 test cases


In [60]:
def run_prompt(prompt_inputs):
    prompt = f"""
Extract key topics mentioned from a passage of text from a scholarly journal into a JSON array of strings.

<text>
{prompt_inputs["content"]}
</text>

Follow these steps:
1. Closely examine the provided text
2. Identify each topic mentioned
3. Add each topic to a JSON array
4. Respond with the JSON array. Do not provide any other text or commentary.
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [61]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary
    - Response should contain the JSON array and nothing else
    """,
)

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 8.75
